# 5장 애플리케이션 계층: 유스 케이스에서의 조율

파이썬으로 구현하는 클린 아키텍처 - 5장 애플리케이션 계층: 유스 케이스에서의 조율 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정
# TodoApp 패키지를 import하기 위한 경로 설정 (Colab/로컬 환경 자동 감지)
# 반드시 첫 번째로 실행해 주세요.
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_5/TodoApp'
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

## 개요

이 장에서는 작업 관리 시스템을 예제로 삼아, 애플리케이션 계층을 효과적으로 구현하는 방법을 살펴본다.

이 장에서 다루는 주요 주제:
* 애플리케이션 계층의 역할 이해
* 실사례에서의 인터랙터 구현
* 요청 및 응답 모델 정의

### 00_error_class.py

## 에러 클래스

구현 패턴을 본격적으로 다루기 전에, 애플리케이션 계층의 핵심 개념 하나를 짚고 넘어가야 한다. 바로 결과 타입result type의 활용이다.

In [ ]:
# 에러 클래스 - 애플리케이션 계층의 표준화된 오류 표현
# 유스케이스 실행 결과를 일관되게 전달하기 위한 구조화된 에러 정보
from dataclasses import dataclass
from enum import Enum
from typing import Optional, Any, Self


# 에러 코드 열거형 - 시스템 전체에서 사용하는 표준 오류 유형
class ErrorCode(Enum):
    NOT_FOUND = "NOT_FOUND"
    VALIDATION_ERROR = "VALIDATION_ERROR"
    # 필요에 따라 다른 에러 코드 추가


# 불변(frozen) 에러 데이터 클래스 - 에러 코드, 메시지, 세부 정보를 캡슐화
@dataclass(frozen=True)
class Error:
    """표준화된 에러 정보"""

    code: ErrorCode
    message: str
    details: Optional[dict[str, Any]] = None

    # 팩토리 메서드: "찾을 수 없음" 에러를 간편하게 생성
    @classmethod
    def not_found(cls, entity: str, entity_id: str) -> Self:
        return cls(
            code=ErrorCode.NOT_FOUND,
            message=f"ID가 {entity_id}인 {entity}을(를) 찾을 수 없음",
        )

    # 팩토리 메서드: "유효성 검증 실패" 에러를 간편하게 생성
    @classmethod
    def validation_error(cls, message: str) -> Self:
        return cls(code=ErrorCode.VALIDATION_ERROR, message=message)

### 02_error_result.py

## 결과 타입(Result Pattern)

다음으로 성공 값이나 오류 중 하나를 담는 Result 클래스를 정의한다.

결과 타입을 사용하면 도메인 연산을 깔끔하게 조율할 수 있다. 다음 예제를 ﻿살펴보자.

In [ ]:
# 결과 타입(Result Pattern) - 성공 또는 실패를 명시적으로 표현하는 Either 패턴
# 예외 대신 반환값으로 성공/실패를 전달하여 흐름 제어를 명확하게 하는 기법
from dataclasses import dataclass
from typing import Any, Optional, Self

from todo_app.application.common.result import Error


@dataclass(frozen=True)
class Result:
    """유스 케이스 실행의 성공 또는 실패를 나타냄"""

    value: Any = None         # 성공 시 결과값
    error: Optional[Error] = None  # 실패 시 에러 정보

    # 성공 여부 확인 - error가 None이면 성공
    @property
    def is_success(self) -> bool:
        return self.error is None

    # 팩토리 메서드: 성공 결과 생성
    @classmethod
    def success(cls, value: Any) -> Self:
        return cls(value=value)

    # 팩토리 메서드: 실패 결과 생성
    @classmethod
    def failure(cls, error: Error) -> Self:
        return cls(error=error)


"""
이 결과 패턴은 도메인 연산을 깔끔하게 조율할 수 있게 해준다. 다음 사용 예제를 살펴보자:

try:
    project = find_project(project_id)
   	task = create_task(task_details)
    project.add_task(task)
   	notify_stakeholders(task)
    return Result.success(TaskResponse.from_entity(task))
except ProjectNotFoundError:
   	return Result.failure(Error.not_found("Project", str(project_id)))
except ValidationError as e:
   	return Result.failure(Error.validation_error(str(e)))
"""

### 02_complete_task_use_case.py

## 작업 완료 유스 케이스

잘 설계된 유스 케이스 인터랙터는 아키텍처 경계를 깔끔하게 유지하면서 도메인 객체를 조율한다. 어떻게 만들 수 있는지 ﻿살펴보자.

먼저 유스 케이스의 외부 구조를 보면 몇 가지 핵심 구성 요소가 눈에 띈다. 의존성 인터페이스가 주입되어 있고, 클래스에는 Result 객체를 반환하는 public execute 메서드가 있다.
다음으로 execute 메서드를 ﻿살펴보자.

In [ ]:
# 작업 완료 유스케이스 - 애플리케이션 계층의 핵심 구성 요소
# 도메인 객체를 조율하면서 아키텍처 경계를 깔끔하게 유지하는 인터랙터
from dataclasses import dataclass
from typing import Optional
from uuid import UUID

from todo_app.application.common.result import Result, Error
from todo_app.application.repositories.task_repository import (
    TaskRepository,
)
from todo_app.domain.exceptions import (
    TaskNotFoundError,
    ValidationError,
)


@dataclass(frozen=True)
class CompleteTaskUseCase:
    """작업을 완료로 표시하고 이해관계자에게 알리는 유스 케이스"""

    # 의존성 주입: 리포지토리 인터페이스에 의존 (구체 구현체가 아닌 추상화에 의존)
    task_repository: TaskRepository

    # execute 메서드: Result 객체를 반환하는 유스케이스의 진입점
    def execute(
        self,
        task_id: UUID,
        completion_notes: Optional[str] = None,
    ) -> Result:

        try:
            # 1단계: 리포지토리에서 작업 조회
            task = self.task_repository.get(task_id)
            # 2단계: 도메인 엔터티의 비즈니스 로직 실행 (상태 전이)
            task.complete(notes=completion_notes)
            # 3단계: 변경된 엔터티를 리포지토리에 저장
            self.task_repository.save(task)

            # 성공 결과 반환 - 단순화된 작업 데이터
            return Result.success(
                {
                    "id": str(task.id),
                    "status": "completed",
                    "completion_date": task.completed_at.isoformat(),
                }
            )

        # 도메인 예외를 Result 패턴으로 변환 - 예외가 계층 경계를 넘지 않도록 처리
        except TaskNotFoundError:
            return Result.failure(Error.not_found("Task", str(task_id)))
        except ValidationError as e:
            return Result.failure(Error.validation_error(str(e)))

### 03_task_repository.py

## 작업 리포지토리 인터페이스

앞서 의존성 주입이 유스 케이스에서 아키텍처 경계를 깔끔하게 유지하는 데 어떻게 도움이 되는지 살펴보았다. 이제 의존성 주입의 장점을 최대한 살리면서 유스 케이스의 유연성과 테스트를 쉽게 하려면 인터페이스를 어떻게 구성해야 하는지 알아보자.

In [ ]:
# 리포지토리 및 서비스 인터페이스 - 애플리케이션 계층에서 정의하는 포트(Port)
# 인프라 계층이 구현해야 할 계약을 추상 클래스로 명시
from abc import abstractmethod, ABC
from uuid import UUID

from todo_app.domain.entities.task import Task


# 작업 리포지토리 인터페이스 - 데이터 접근에 대한 추상화
# 애플리케이션 계층이 필요로 하는 CRUD 연산만 정의
class TaskRepository(ABC):
    """애플리케이션 계층에서 정의한 리포지토리 인터페이스"""

    @abstractmethod
    def get(self, task_id: UUID) -> Task:
        """ID로 작업을 조회함"""
        pass

    @abstractmethod
    def save(self, task: Task) -> None:
        """작업을 리포지토리에 저장함"""
        pass

    @abstractmethod
    def delete(self, task_id: UUID) -> None:
        """리포지토리에서 작업을 삭제함"""
        pass


# 알림 서비스 인터페이스 - 외부 알림 시스템에 대한 추상화
class NotificationService(ABC):
    """알림 전송을 위한 서비스 인터페이스"""

    @abstractmethod
    def notify_task_assigned(self, task_id: UUID) -> None:
        """작업이 할당되었을 때 알림"""
        pass

    @abstractmethod
    def notify_task_completed(self, task_id: UUID) -> None:
        """작업이 완료되었을 때 알림"""
        pass

### 04_mongodb_repository.py

## MongoDB 리포지토리 구현

애플리케이션 계층에 정의한 `TaskRepository` 인터페이스를 MongoDB로 구현하는 인프라 계층 어댑터이다. 인터페이스가 필요한 기능만 정확히 정의하므로 아키텍처 경계가 강화된다.

In [ ]:
# MongoDB 리포지토리 구현체 - 인프라 계층의 어댑터
# 애플리케이션 계층의 TaskRepository 인터페이스를 MongoDB로 구현
from uuid import UUID

from todo_app.application.repositories.task_repository import (
    TaskRepository,
)
from todo_app.domain.entities.task import Task
from todo_app.domain.exceptions import TaskNotFoundError


class MongoClient:
    def __init__(self):
        self.task_management = None

    """mypy를 위한 스텁 클래스"""

    ...


# TaskRepository 인터페이스의 MongoDB 구현체 (외부 원의 어댑터)
class MongoDbTaskRepository(TaskRepository):
    """TaskRepository 인터페이스의 MongoDB 구현체"""

    def __init__(self, client: MongoClient):
        # MongoDB 클라이언트 주입 - 인프라 세부사항
        self.client = client
        self.db = client.task_management
        self.tasks = self.db.tasks

    def get(self, task_id: UUID) -> Task:
        """ID로 작업을 조회함 - MongoDB 쿼리로 구현"""
        document = self.tasks.find_one({"_id": str(task_id)})
        if not document:
            raise TaskNotFoundError(task_id)
        # ... 나머지 메서드 구현

    # 다른 인터페이스 메서드 구현 ...

### 05_init_complete_project_use_case.py

## 프로젝트 완료 유스 케이스 초기화

인터페이스를 애플리케이션 계층에 정의하면 아키텍처 경계가 강화되고, 바깥 계층이 구현할 명확한 계약이 마련된다. 테스트 시에도 각 유스케이스가 요구하는 부분에만 집중할 수 있다.

In [ ]:
"""
18페이지의 NotificationService 초기 예제
"""

# 프로젝트 완료 유스케이스 - 여러 도메인 객체를 조율하는 복합 유스케이스
from dataclasses import dataclass
from typing import Optional
from uuid import UUID

from todo_app.application.common.result import Result, Error
from todo_app.application.repositories.project_repository import (
    ProjectRepository,
)
from todo_app.application.repositories.task_repository import (
    TaskRepository,
)
from todo_app.domain.exceptions import (
    ProjectNotFoundError,
    ValidationError,
)

from todo_app.domain.entities.task import Task  # [수정] domain.models.task → domain.entities.task


class NotificationService:
    """mypy를 위한 스텁"""

    def notify_task_completed(self, task: Task) -> None:
        pass


@dataclass(frozen=True)
class CompleteProjectUseCase:
    # 의존성 주입: 3개의 인터페이스(포트)에 의존
    project_repository: ProjectRepository
    task_repository: TaskRepository
    notification_service: NotificationService

    def execute(self, project_id: UUID, completion_notes: Optional[str] = None) -> Result:
        try:
            # 1단계: 프로젝트 존재 여부 검증
            project = self.project_repository.get(project_id)

            # 2단계: 미완료된 모든 작업을 순회하며 완료 처리
            for task in project.incomplete_tasks:
                task.complete()
                self.task_repository.save(task)
                # 각 작업 완료 시 이해관계자에게 알림 전송
                self.notification_service.notify_task_completed(task)

            # 3단계: 프로젝트 자체를 완료 처리
            project.mark_completed(notes=completion_notes)
            self.project_repository.save(project)

            # 성공 결과 반환 - 프로젝트 요약 정보
            return Result.success(
                {
                    "id": str(project.id),
                    "status": project.status,
                    "completion_date": project.completed_at,
                    "task_count": len(project.tasks),
                    "completion_notes": project.completion_notes,
                }
            )

        # 도메인 예외를 Result 패턴으로 변환
        except ProjectNotFoundError:
            return Result.failure(Error.not_found("Project", str(project_id)))
        except ValidationError as e:
            return Result.failure(Error.validation_error(str(e)))

### 06_complete_project_request.py

## 프로젝트 완료 요청 DTO

요청 모델은 외부 입력을 검증하고 도메인 타입으로 변환하는 경계 객체이다. `__post_init__`에서 사전 검증을 수행하고, `to_execution_params()`로 API 형식(문자열 ID)을 도메인 타입(UUID)으로 변환하여 유스케이스가 비즈니스 로직에만 집중할 수 있게 한다.

In [ ]:
# 요청 DTO(Data Transfer Object) - 외부 입력을 검증하고 도메인 타입으로 변환
# API 경계에서 들어오는 데이터를 유스케이스가 기대하는 형식으로 변환하는 경계 객체
from dataclasses import dataclass
from typing import Optional
from uuid import UUID

from todo_app.domain.exceptions import ValidationError


@dataclass(frozen=True)
class CompleteProjectRequest:
    """프로젝트 완료 요청을 위한 데이터 구조"""

    project_id: str  # API에서 문자열로 전달됨 (내부에서 UUID로 변환)
    completion_notes: Optional[str] = None

    # 생성 시 자동 유효성 검증 - 잘못된 데이터의 유입을 경계에서 차단
    def __post_init__(self) -> None:
        """요청 데이터 검증"""
        if not self.project_id.strip():
            raise ValidationError("프로젝트 ID는 필수입니다")

        if self.completion_notes and len(self.completion_notes) > 1000:
            raise ValidationError(
                "완료 메모 1000자 초과 불가능"
            )

    # API 형식(문자열)을 도메인 타입(UUID)으로 변환하는 경계 변환 메서드
    def to_execution_params(self) -> dict:
        """검증된 요청 데이터를 유스 케이스 매개변수로 변환"""
        return {
            "project_id": UUID(self.project_id),
            "completion_notes": self.completion_notes,
        }

### 07_complete_project_response.py

## 프로젝트 완료 응답 DTO

응답 모델은 도메인 객체를 외부에 노출하기 적합한 구조로 변환한다. `from_entity()` 메서드가 도메인 엔터티를 직렬화 가능한 형식으로 매핑하며, UUID→문자열 변환이나 `task_count` 같은 파생 필드 추가를 담당한다. 요청 DTO의 `to_execution_params()`와 대칭적 구조를 이룬다.

In [ ]:
# 응답 DTO - 도메인 객체를 외부에 노출하기 적합한 구조로 변환
# from_entity 메서드로 도메인 엔터티를 직렬화 가능한 형식으로 매핑
from dataclasses import dataclass
from typing import Optional, Self

from todo_app.domain.entities.project import Project


class UserService:
    """mypy를 위한 스텁"""

    ...


@dataclass(frozen=True)
class CompleteProjectResponse:
    """프로젝트 완료 응답을 위한 데이터 구조"""

    id: str                        # UUID → 문자열로 변환된 프로젝트 식별자
    status: str                    # 열거형 → 문자열로 변환된 상태값
    completion_date: str           # datetime → ISO 문자열로 변환된 완료 일시
    task_count: int                # 파생 필드 - 도메인 객체에서 계산
    completion_notes: Optional[str]

    # 팩토리 메서드: 도메인 엔터티 → 응답 DTO 변환 (나가는 방향의 경계 변환)
    @classmethod
    def from_entity(cls, project: Project, user_service: UserService) -> Self:
        """도메인 엔터티로부터 응답 생성"""
        return cls(
            id=str(project.id),
            status=project.status.value,
            completion_date=project.completed_at.isoformat(),
            task_count=len(project.tasks),
            completion_notes=project.completion_notes,
        )

### 08_evolved_complete_project_use_case.py

## 발전된 프로젝트 완료 유스 케이스

요청 DTO의 `to_execution_params()`로 입력을 변환하고, 응답 DTO의 `from_entity()`로 출력을 변환하여 유스케이스가 순수하게 도메인 객체만 다루도록 한다.

In [ ]:
# 발전된 프로젝트 완료 유스케이스 - 요청/응답 DTO와 포트를 모두 활용
# 경계 메커니즘(DTO, 포트)이 함께 작동하여 아키텍처 경계를 깔끔하게 유지
from dataclasses import dataclass

from todo_app.domain.entities.task import Task
from todo_app.application.common.result import Result, Error
from todo_app.application.dtos.project_dtos import (
    CompleteProjectRequest,
    CompleteProjectResponse,
)
from todo_app.application.repositories.project_repository import (
    ProjectRepository,
)
from todo_app.application.repositories.task_repository import (
    TaskRepository,
)
from todo_app.domain.exceptions import (
    ValidationError,
    ProjectNotFoundError,
)


class NotificationService:
    """mypy를 위한 스텁"""

    def notify_task_completed(self, task: Task) -> None:
        pass


@dataclass(frozen=True)
class CompleteProjectUseCase:
    project_repository: ProjectRepository
    task_repository: TaskRepository
    notification_service: NotificationService

    # 요청 DTO를 매개변수로 받아 경계 변환 → 도메인 로직 → 응답 DTO 생성
    def execute(self, request: CompleteProjectRequest) -> Result:
        try:
            # 요청 DTO의 to_execution_params()로 API 형식 → 도메인 타입 변환
            params = request.to_execution_params()
            project = self.project_repository.get(params["project_id"])
            project.mark_completed(notes=params["completion_notes"])

            # 미완료된 모든 작업 완료 처리
            # ... 간결함을 위해 생략

            self.project_repository.save(project)

            # 응답 DTO의 from_entity()로 도메인 엔터티 → API 형식 변환
            response = CompleteProjectResponse.from_entity(project)
            return Result.success(response)

        except ProjectNotFoundError:
            return Result.failure(Error.not_found("Project", str(params["project_id"])))
        except ValidationError as e:
            return Result.failure(Error.validation_error(str(e)))

### 09_notification_port.py

## 알림 포트(Port)

어댑터 패턴으로 서드파티 서비스의 인터페이스를 기존 포트에 맞춘다. 서비스 교체/업그레이드 시 어댑터의 변환 계층만 수정하면 되므로, 유스케이스와 아키텍처 경계를 깔끔하게 유지할 수 있다.

In [ ]:
# 알림 포트(Port) - 애플리케이션 계층이 필요로 하는 외부 서비스 기능 정의
# 구체적 구현(이메일, SMS 등)을 알지 못한 채 기능만 명시하는 추상 인터페이스
from abc import abstractmethod, ABC
from uuid import UUID

from todo_app.domain.entities.task import Task  # [보완] Task import 추가


# NotificationPort: 유스케이스가 필요로 하는 알림 기능의 계약
# 인프라 계층에서 이 인터페이스를 구현하여 실제 알림 전송
class NotificationPort(ABC):

    @abstractmethod
    def notify_task_completed(self, task: Task) -> None:
        """작업이 완료되었을 때 알림"""
        pass

    # 필요에 따라 추가 기능 확장

### 10_set_task_priority_use_case.py

## 작업 우선순위 설정 유스 케이스

구체 구현체가 아닌 추상 포트(`NotificationPort`)에 의존하여 의존성 규칙을 준수한다. 경계 메커니즘의 역할 분담:
- **요청/응답 모델**: API 경계에서 데이터 변환
- **포트**: 유스케이스에 필요한 서비스 기능 정의
- **유스케이스**: 전체 흐름 조율

In [ ]:
# 작업 우선순위 설정 유스케이스 - 포트를 활용한 알림 연동
# 추상 인터페이스(NotificationPort)에 의존하여 구현체 교체 가능
from dataclasses import dataclass

from todo_app.application.common.result import Result, Error
from todo_app.application.dtos.task_dtos import (
    TaskResponse,
    SetTaskPriorityRequest,
)
from todo_app.application.service_ports.notifications import (
    NotificationPort,
)
from todo_app.application.repositories.task_repository import (
    TaskRepository,
)
from todo_app.domain.exceptions import ValidationError
from todo_app.domain.value_objects import Priority


@dataclass
class SetTaskPriorityUseCase:
    task_repository: TaskRepository
    notification_service: NotificationPort  # 포트(추상 인터페이스)에 의존

    def execute(self, request: SetTaskPriorityRequest) -> Result:
        try:
            # 요청 DTO → 도메인 타입 변환
            params = request.to_execution_params()

            task = self.task_repository.get(params["task_id"])
            task.priority = params["priority"]

            self.task_repository.save(task)

            # 비즈니스 규칙: HIGH 우선순위 작업은 알림 발송
            if task.priority == Priority.HIGH:
                self.notification_service.notify_task_high_priority(task)

            return Result.success(TaskResponse.from_entity(task))
        except ValidationError as e:
            return Result.failure(Error.validation_error(str(e)))

### 11_task_management_use_case.py

## 작업 관리 유스 케이스

시스템이 발전하면서 신기능을 추가하고 변화하는 서비스 구현에 적응할 수 있는 패턴이 필요해진다. 이런 변화를 관리하는 두 가지 핵심 패턴을 ﻿살펴보자.

선택적 서비스를 등록

In [ ]:
# 작업 관리 유스케이스 - 선택적 서비스 등록 패턴
# 필수 서비스(알림)와 선택적 서비스(분석, 감사)를 구분하여 유연성 확보
from dataclasses import field, dataclass
from typing import Any
from uuid import UUID

from todo_app.application.common.result import Result, Error
from todo_app.application.dtos.task_dtos import TaskResponse
from todo_app.application.service_ports.notifications import (  # [수정] serviceports → service_ports
    NotificationPort,
)
from todo_app.application.repositories.task_repository import (
    TaskRepository,
)
from todo_app.domain.exceptions import ValidationError


@dataclass(frozen=True)
class TaskManagementUseCase:
    task_repository: TaskRepository
    notification_service: NotificationPort  # 필수 의존성
    _optional_services: dict[str, Any] = field(default_factory=dict)  # 선택적 서비스 저장소

    # 선택적 서비스 등록 - 분석, 감사 등 추가 기능을 런타임에 연결
    def register_service(self, name: str, service: Any) -> None:
        """선택적 서비스를 등록"""
        self._optional_services[name] = service

    def complete_task(self, task_id: UUID) -> Result:
        try:
            task = self.task_repository.get(task_id)
            task.complete()
            self.task_repository.save(task)

            # 필수 알림 처리 - NotificationPort를 통한 알림 전송
            self.notification_service.notify_task_completed(task)

            # 선택적 연동 서비스 처리 - 등록된 서비스만 실행
            if analytics := self._optional_services.get("analytics"):
                analytics.track_task_completion(task.id)
            if audit := self._optional_services.get("audit"):
                audit.log_task_completion(task.id)

            return Result.success(TaskResponse.from_entity(task))
        except ValidationError as e:
            return Result.failure(Error.validation_error(str(e)))

### 12_adapting_to_service_change.py

## 서비스 변경에 대한 적응

어댑터 패턴으로 서드파티 서비스의 인터페이스를 기존 포트에 맞춘다. 서비스 교체/업그레이드 시 어댑터의 변환 계층만 수정하면 되므로, 유스케이스에는 영향이 없다.

In [ ]:
# 어댑터 패턴 - 서로 다른 인터페이스를 가진 외부 서비스를 기존 포트에 연결
# 서드파티 서비스 교체 시 어댑터만 수정하면 유스케이스는 변경 불필요
from uuid import UUID

from todo_app.application.service_ports.notifications import (
    NotificationPort,
)


# 서드파티 알림 서비스 - 자체 인터페이스(send_notification)를 가진 외부 라이브러리
class ModernNotificationService:
    """서로 다른 인터페이스를 가진 외부(서드파티) 알림 서비스"""

    def send_notification(self, payload: dict) -> None:
        # 최신 알림 서비스의 실제 구현
        pass


# 어댑터: NotificationPort 인터페이스 → ModernNotificationService 연결
# 기존 애플리케이션의 포트 인터페이스와 외부 서비스의 인터페이스 간 변환 계층
class ModernNotificationAdapter(NotificationPort):
    """기존 애플리케이션의 알림 인터페이스에 맞게 최신 알림 서비스를 연결하기 위한 어댑터"""

    def __init__(self, modern_service: ModernNotificationService):
        self._service = modern_service

    # 포트의 notify_task_completed → 외부 서비스의 send_notification으로 변환
    def notify_task_completed(self, task_id: UUID) -> None:
        self._service.send_notification(
            {"type": "TASK_COMPLETED", "taskId": str(task_id)}
        )